In [ ]:
# Databricks notebook source
# COMMAND ----------
# MAGIC %md
# MAGIC # Customer 360 Profile Creation
# MAGIC This notebook processes insurance data to create a comprehensive customer 360 profile. It involves loading data from CSV files, performing data transformations, and writing the final output to a Unity Catalog table.

# COMMAND ----------
# MAGIC
# Import necessary libraries
import logging
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from datetime import datetime

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# COMMAND ----------
# MAGIC
# Define file paths for CSVs
policy_csv_path = "tfs://dataeconomy-9k42/62457/uploads/62457/29aabe9d-c354-4176-8f98-2dd7a5fd7216/policy.csv"
claims_csv_path = "tfs://dataeconomy-9k42/62457/uploads/62457/29aabe9d-c354-4176-8f98-2dd7a5fd7216/claims.csv"
demographics_csv_path = "tfs://dataeconomy-9k42/62457/uploads/62457/29aabe9d-c354-4176-8f98-2dd7a5fd7216/demographics.csv"
scores_csv_path = "tfs://dataeconomy-9k42/62457/uploads/62457/29aabe9d-c354-4176-8f98-2dd7a5fd7216/scores.csv"
aiml_insights_csv_path = "tfs://dataeconomy-9k42/62457/uploads/62457/29aabe9d-c354-4176-8f98-2dd7a5fd7216/aiml_insights.csv"

# COMMAND ----------
# MAGIC
try:
    # Load data from CSVs into DataFrames
    policy_df = spark.read.csv(policy_csv_path, header=True, inferSchema=True)
    claims_df = spark.read.csv(claims_csv_path, header=True, inferSchema=True)
    demographics_df = spark.read.csv(demographics_csv_path, header=True, inferSchema=True)
    scores_df = spark.read.csv(scores_csv_path, header=True, inferSchema=True)
    aiml_insights_df = spark.read.csv(aiml_insights_csv_path, header=True, inferSchema=True)

    logger.info("Data loaded successfully from CSV files.")
except Exception as e:
    logger.error(f"An error occurred while loading data: {e}")

# COMMAND ----------
# MAGIC
try:
    # Select relevant fields from demographics
    selected_demographics_df = demographics_df.select(
        "Customer_ID", "Customer_Name", "Email", "Phone_Number", "Address", "City", "State", "Postal_Code",
        "Date_of_Birth", "Gender", "Marital_Status", "Occupation", "Income_Level", "Customer_Segment"
    )

    # Join demographics with policy data on Customer_ID
    joined_df = selected_demographics_df.join(
        policy_df, F.col("Customer_ID") == F.col("customer_id"), "inner"
    ).join(
        claims_df, F.col("policy_id") == F.col("Policy_ID"), "inner"
    )

    logger.info("Data joined successfully.")
except Exception as e:
    logger.error(f"An error occurred during data joining: {e}")

# COMMAND ----------
# MAGIC
try:
    # Aggregate data
    aggregated_df = joined_df.groupBy("Customer_ID").agg(
        F.count("Claim_ID").alias("Total_Claims"),
        F.count("policy_id").alias("Policy_Count"),
        F.max("Claim_Date").alias("Recent_Claim_Date"),
        F.avg("Claim_Amount").alias("Average_Claim_Amount")
    )

    logger.info("Data aggregated successfully.")
except Exception as e:
    logger.error(f"An error occurred during data aggregation: {e}")

# COMMAND ----------
# MAGIC
try:
    # Custom calculations
    age_expr = F.floor(F.datediff(F.current_date(), F.to_date("Date_of_Birth", 'yyyy-MM-dd')) / 365)
    claim_to_premium_ratio_expr = F.expr("Claim_Amount / total_premium_paid")
    claims_per_policy_expr = F.expr("Total_Claims / Policy_Count")

    final_df = aggregated_df.withColumn("Age", age_expr) \
                            .withColumn("Claim_To_Premium_Ratio", claim_to_premium_ratio_expr) \
                            .withColumn("Claims_Per_Policy", claims_per_policy_expr) \
                            .withColumn("Retention_Rate", F.lit(0.85)) \
                            .withColumn("Cross_Sell_Opportunities", F.lit("Multi-Policy Discount, Home Coverage Add-on")) \
                            .withColumn("Upsell_Potential", F.lit("Premium Vehicle Coverage"))

    logger.info("Custom calculations applied successfully.")
except Exception as e:
    logger.error(f"An error occurred during custom calculations: {e}")

# COMMAND ----------
# MAGIC
try:
    # Join with scores and AI/ML insights
    customer_360_df = final_df.join(scores_df, "Customer_ID", "inner").join(aiml_insights_df, "Customer_ID", "inner")

    logger.info("Final customer 360 profile created successfully.")
except Exception as e:
    logger.error(f"An error occurred during final profile creation: {e}")

# COMMAND ----------
# MAGIC
try:
    # Write the final DataFrame to Unity Catalog table
    spark.sql("DROP TABLE IF EXISTS catalog.target_db.Customer_360")
    customer_360_df.write.format("delta").mode("overwrite").saveAsTable("catalog.target_db.Customer_360")

    logger.info("Data written to Unity Catalog table successfully.")
except Exception as e:
    logger.error(f"An error occurred while writing data to Unity Catalog: {e}")
